In [ ]:
# For Colab
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import pandas_gbq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

PROJECT_ID = "robust-solution-470105-q7"   # Replace with your GCP project ID
pandas_gbq.context.project = PROJECT_ID
print("Setup complete")

Mounted at /content/drive
Setup complete


In [ ]:
T = 48
HOURS = np.arange(T)

LAB_FEATURES = ['creatinine', 'glucose', 'sodium', 'potassium', 'hematocrit', 'wbc']
VITAL_FEATURES = ['heart_rate', 'sbp', 'dbp', 'mbp', 'respiratory_rate', 'temperature', 'spo2']
DEMO_FEATURES = ['age', 'gender']
ALL_FEATURES = LAB_FEATURES + VITAL_FEATURES + DEMO_FEATURES

# Itemid mappings
LAB_ITEMIDS = {
    50912: 'creatinine',   # creatinine
    50931: 'glucose',      # glucose
    50983: 'sodium',       # sodium
    50971: 'potassium',    # potassium
    51221: 'hematocrit',   # hematocrit
    51301: 'wbc'           # WBC
}

VITAL_ITEMIDS = {
    220045: 'heart_rate',
    220179: 'sbp',
    220180: 'dbp',
    220181: 'mbp',
    220210: 'respiratory_rate',
    223761: 'temperature',   # Celsius
    223762: 'temperature',
    220277: 'spo2'
}

In [ ]:
query_icu = """
WITH first_icu_stays AS (
  SELECT
    stay_id, hadm_id, subject_id, intime, outtime,
    ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime) AS stay_number
  FROM `physionet-data.mimiciv_3_1_icu.icustays`
),
first_stays AS (
  SELECT stay_id, hadm_id, subject_id, intime, outtime
  FROM first_icu_stays
  WHERE stay_number = 1
),
patient_info AS (
  SELECT
    p.subject_id, p.dod, p.gender,
    a.hadm_id, a.hospital_expire_flag,
    DATE_DIFF(fs.intime, PARSE_DATE('%Y', CAST(p.anchor_year AS STRING)), YEAR) AS age
  FROM `physionet-data.mimiciv_3_1_hosp.patients` p
  JOIN `physionet-data.mimiciv_3_1_hosp.admissions` a ON p.subject_id = a.subject_id
  JOIN first_stays fs ON a.hadm_id = fs.hadm_id
)
SELECT
  fs.stay_id, fs.hadm_id, fs.subject_id, fs.intime, fs.outtime,
  pi.dod, pi.gender, pi.age, pi.hospital_expire_flag,
  CASE
    WHEN pi.hospital_expire_flag = 1 THEN 1
    WHEN pi.dod IS NOT NULL AND pi.dod <= fs.outtime THEN 1
    ELSE 0
  END AS mortality
FROM first_stays fs
LEFT JOIN patient_info pi ON fs.hadm_id = pi.hadm_id
"""

df_icu = pandas_gbq.read_gbq(query_icu, project_id=PROJECT_ID)
df_icu['intime'] = pd.to_datetime(df_icu['intime'])
df_icu['outtime'] = pd.to_datetime(df_icu['outtime'])
df_icu['dod'] = pd.to_datetime(df_icu['dod'])

print(f"First ICU stays: {len(df_icu)}")
print(f"Mortality rate: {df_icu['mortality'].mean()*100:.1f}%")
df_icu.head()

Downloading: 100%|██████████|
First ICU stays: 65366
Mortality rate: 11.0%


,stay_id,hadm_id,subject_id,intime,outtime,dod,gender,age,hospital_expire_flag,mortality
0,39553978,29079034,10000032,2180-07-23 14:00:00,2180-07-23 23:50:47,2180-09-09,F,0,0,0
1,37081114,25860671,10000690,2150-11-02 19:37:00,2150-11-06 17:03:17,2152-01-30,F,0,0,0
2,39765666,26913865,10000980,2189-06-27 08:42:00,2189-06-27 20:38:27,2193-08-26,F,3,0,0
3,37067082,24597018,10001217,2157-11-20 19:18:02,2157-11-21 22:08:00,NaT,F,0,0,0
4,31205490,25563031,10001725,2110-04-11 15:52:22,2110-04-12 23:59:56,NaT,F,0,0,0


In [ ]:
query_labs = """
WITH first_stays AS (
  SELECT stay_id, hadm_id, intime
  FROM (
    SELECT stay_id, hadm_id, intime,
           ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime) AS stay_number
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
  )
  WHERE stay_number = 1
)
SELECT
  s.stay_id,
  l.itemid,
  TIMESTAMP_DIFF(l.charttime, s.intime, HOUR) AS hours_in,
  AVG(l.valuenum) AS valuenum
FROM `physionet-data.mimiciv_3_1_hosp.labevents` l
INNER JOIN first_stays s ON l.hadm_id = s.hadm_id
WHERE l.valuenum IS NOT NULL
  AND l.itemid IN (50912, 50931, 50983, 50971, 51221, 51301)
  AND l.charttime BETWEEN s.intime AND TIMESTAMP_ADD(s.intime, INTERVAL 48 HOUR)
GROUP BY s.stay_id, l.itemid, TIMESTAMP_DIFF(l.charttime, s.intime, HOUR)
"""

df_labs = pandas_gbq.read_gbq(query_labs, project_id=PROJECT_ID)
df_labs['feature'] = df_labs['itemid'].map(LAB_ITEMIDS)
df_labs = df_labs.drop('itemid', axis=1)

print(f"Lab rows: {len(df_labs)}")
df_labs.head()

Downloading: 100%|██████████|
Lab rows: 1319028


,stay_id,hours_in,valuenum,feature
0,35987620,0,3.8,potassium
1,32975173,0,28.3,hematocrit
2,34307117,0,126.0,sodium
3,30704941,0,10.4,wbc
4,34156114,0,137.0,sodium


In [ ]:
query_vitals = """
WITH first_stays AS (
  SELECT stay_id, intime
  FROM (
    SELECT stay_id, intime,
           ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime) AS stay_number
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
  )
  WHERE stay_number = 1
)
SELECT
  s.stay_id,
  ce.itemid,
  TIMESTAMP_DIFF(ce.charttime, s.intime, HOUR) AS hours_in,
  AVG(ce.valuenum) AS valuenum
FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
INNER JOIN first_stays s ON ce.stay_id = s.stay_id
WHERE ce.valuenum IS NOT NULL
  AND ce.itemid IN (220045, 220179, 220180, 220181, 220210, 223761, 223762, 220277)
  AND ce.charttime BETWEEN s.intime AND TIMESTAMP_ADD(s.intime, INTERVAL 48 HOUR)
GROUP BY s.stay_id, ce.itemid, TIMESTAMP_DIFF(ce.charttime, s.intime, HOUR)
"""

df_vitals = pandas_gbq.read_gbq(query_vitals, project_id=PROJECT_ID)
df_vitals['feature'] = df_vitals['itemid'].map(VITAL_ITEMIDS)
df_vitals = df_vitals.drop('itemid', axis=1)

print(f"Vitals rows: {len(df_vitals)}")
df_vitals.head()

Downloading: 100%|█████████▉|

In [ ]:
T = 48  # first 48 hours

# EDIT THESE IF YOUR COLUMN NAMES ARE DIFFERENT
ID_COL = "stay_id"
TIME_COL = "charttime"
FEATURE_COL = "feature"     # sometimes called "label" or "item"
VALUE_COL = "valuenum"
INTIME_COL = "intime"
LABEL_COL = "mortality"     # 0/1 label in df_icu
AGE_COL = "age"
GENDER_COL = "gender"

# output folder
OUT_DIR = "mimic_numpy"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
print("df_vitals columns:", df_vitals.columns.tolist())
print("df_labs columns:", df_labs.columns.tolist())
print("df_icu columns:", df_icu.columns.tolist())

In [ ]:
# copy
vitals = df_vitals.copy()
labs   = df_labs.copy()
icu    = df_icu.copy()

icu_meta = icu[[ID_COL, LABEL_COL, AGE_COL, GENDER_COL]].drop_duplicates(subset=[ID_COL])

vitals = vitals.merge(icu_meta, on=ID_COL, how="left")
labs   = labs.merge(icu_meta, on=ID_COL, how="left")

# combine
meas = pd.concat([vitals, labs], ignore_index=True)

# use hours_in
meas["hours_in"] = meas["hours_in"].astype(int)
meas = meas[(meas["hours_in"] >= 0) & (meas["hours_in"] < 48)]

In [ ]:
print("Combined measurement shape:", meas.shape)
print("Unique stays:", meas[ID_COL].nunique())
print("Unique features:", meas[FEATURE_COL].nunique())

In [ ]:
# Dynamic features are whatever is present in the measurement tables,
# excluding static ones if they accidentally appear there.
dynamic_features = sorted(
    set(meas[FEATURE_COL].astype(str).unique()) - {AGE_COL, GENDER_COL}
)

print("Dynamic features:", dynamic_features)
print("Count:", len(dynamic_features))

# Static features are repeated across time
static_features = [AGE_COL, GENDER_COL]

# Final feature order: dynamic first, static last
all_features = dynamic_features + static_features

print("Final feature count:", len(all_features))

In [ ]:
def encode_gender(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        x = x.strip().upper()
        if x in {"M", "MALE", "1"}:
            return 1.0
        if x in {"F", "FEMALE", "0"}:
            return 0.0
    try:
        return float(x)
    except Exception:
        return np.nan


def forward_fill_1d(arr):
    """
    Forward fill 1D array of length T.
    Keeps NaNs at the beginning if no previous value exists.
    """
    out = arr.copy()
    last = np.nan
    for t in range(len(out)):
        if np.isnan(out[t]):
            out[t] = last
        else:
            last = out[t]
    return out

In [ ]:
def forward_fill_dynamic_block(X, dynamic_count):
    """
    X shape: (N, T, F)
    Forward fill only the dynamic features.
    Static features are left as is.
    """
    out = X.copy()
    N, T_, F = out.shape

    for n in range(N):
        for j in range(dynamic_count):
            out[n, :, j] = forward_fill_1d(out[n, :, j])

    return out


def fill_remaining_nans_with_train_means(X, train_means):
    """
    Replace remaining NaNs with training means.
    """
    out = X.copy()
    nan_pos = np.where(np.isnan(out))
    out[nan_pos] = np.take(train_means, nan_pos[2])
    return out


def standardize_with_train_stats(X_train, X_val, X_test):
    """
    Standardize feature-wise using train statistics.
    """
    train_flat = X_train.reshape(-1, X_train.shape[-1])
    mean = np.nanmean(train_flat, axis=0)
    std = np.nanstd(train_flat, axis=0) + 1e-6

    X_train = (X_train - mean) / std
    X_val   = (X_val   - mean) / std
    X_test  = (X_test  - mean) / std

    return X_train, X_val, X_test, mean, std

In [ ]:
dynamic_count = len(dynamic_features)
F = len(all_features)

X_values_train = X_values[train_idx]
X_values_val   = X_values[val_idx]
X_values_test  = X_values[test_idx]

X_gaps_train = X_gaps[train_idx]
X_gaps_val   = X_gaps[val_idx]
X_gaps_test  = X_gaps[test_idx]

mask_train = mask[train_idx]
mask_val   = mask[val_idx]
mask_test  = mask[test_idx]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

# 1) forward fill dynamic features only
X_values_train = forward_fill_dynamic_block(X_values_train, dynamic_count)
X_values_val   = forward_fill_dynamic_block(X_values_val, dynamic_count)
X_values_test  = forward_fill_dynamic_block(X_values_test, dynamic_count)

# 2) compute train means for imputation
train_flat = X_values_train.reshape(-1, F)
train_means = np.nanmean(train_flat, axis=0)

# 3) fill remaining NaNs with train means
X_values_train = fill_remaining_nans_with_train_means(X_values_train, train_means)
X_values_val   = fill_remaining_nans_with_train_means(X_values_val, train_means)
X_values_test  = fill_remaining_nans_with_train_means(X_values_test, train_means)

# 4) standardize values and gaps separately
X_values_train, X_values_val, X_values_test, value_mean, value_std = standardize_with_train_stats(
    X_values_train, X_values_val, X_values_test
)

X_gaps_train, X_gaps_val, X_gaps_test, gap_mean, gap_std = standardize_with_train_stats(
    X_gaps_train, X_gaps_val, X_gaps_test
)

# 5) concatenate values + gaps
X_train = np.concatenate([X_values_train, X_gaps_train], axis=-1)
X_val   = np.concatenate([X_values_val,   X_gaps_val],   axis=-1)
X_test  = np.concatenate([X_values_test,  X_gaps_test],  axis=-1)

print("Final X_train:", X_train.shape)
print("Final X_val:  ", X_val.shape)
print("Final X_test: ", X_test.shape)
print("mask_train:   ", mask_train.shape)
print("y_train:      ", y_train.shape)

In [ ]:
print("Train mortality:", y_train.mean() * 100)
print("Val mortality:  ", y_val.mean() * 100)
print("Test mortality: ", y_test.mean() * 100)

obs_ratio = mask_train.mean(axis=1)
print("Mean observed ratio (train):", obs_ratio.mean())
print("Min observed ratio (train): ", obs_ratio.min())
print("Max observed ratio (train): ", obs_ratio.max())

gap_block = X_train[:, :, len(all_features):]
print("Gap block mean:", gap_block.mean())
print("Gap block std: ", gap_block.std())
print("Gap block min: ", gap_block.min())
print("Gap block max: ", gap_block.max())

In [ ]:
np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train.astype(np.float32))
np.save(os.path.join(OUT_DIR, "X_val.npy"),   X_val.astype(np.float32))
np.save(os.path.join(OUT_DIR, "X_test.npy"),  X_test.astype(np.float32))

np.save(os.path.join(OUT_DIR, "mask_train.npy"), mask_train.astype(bool))
np.save(os.path.join(OUT_DIR, "mask_val.npy"),   mask_val.astype(bool))
np.save(os.path.join(OUT_DIR, "mask_test.npy"),  mask_test.astype(bool))

np.save(os.path.join(OUT_DIR, "y_train.npy"), y_train.astype(np.float32))
np.save(os.path.join(OUT_DIR, "y_val.npy"),   y_val.astype(np.float32))
np.save(os.path.join(OUT_DIR, "y_test.npy"),  y_test.astype(np.float32))

print("Saved all numpy files to:", OUT_DIR)